# Gold — ETA Accuracy

Analyses how accurate the mo-bi.ro API ETAs are at crossing time.

At each stop we record `eta_before` — the seconds the API said the bus was away
just before we detected the crossing. A perfect API would always give `eta_before = 0`.
In practice, the crossing fires on the poll *after* the bus passes, so `eta_before`
is the detection lag introduced by the poll interval.

This notebook quantifies that lag per stop and looks for systematic bias.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

SILVER = Path("../data/silver_journeys.parquet")
df = pd.read_parquet(SILVER)
print(f"Loaded {len(df):,} journeys")

pd.set_option("display.float_format", "{:.1f}".format)

## Detection lag (eta_before) per stop

In [ ]:
STOPS = [
    ("sincai",           "Gh. Sincai"),
    ("marasesti",        "Bd. Marasesti"),
    ("sf_gheorghe",      "Piata Sf. Gheorghe"),
    ("universitate",     "Universitate"),
    ("nicolae_balcescu", "Bd. Nicolae Balcescu"),
    ("arthur_verona",    "Arthur Verona"),
    ("romana",           "Piata Romana"),
]

rows = []
for key, label in STOPS:
    col = f"{key}_eta_before"
    if col not in df.columns:
        continue
    vals = pd.to_numeric(df[col], errors="coerce").dropna()
    rows.append({
        "stop":   label,
        "mean_s": vals.mean(),
        "median_s": vals.median(),
        "p90_s":  vals.quantile(0.9),
        "count":  len(vals),
    })

lag = pd.DataFrame(rows).set_index("stop")
print(lag.round(1))

In [ ]:
ax = lag[["mean_s", "p90_s"]].plot(kind="bar", figsize=(10, 4),
    title="Detection lag (eta_before) at crossing — lower is more accurate",
    ylabel="seconds")
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
plt.tight_layout()
plt.show()

## eta_before distribution at Gh. Sincai

Sincai has adaptive fast-polling (20s / 10s when ETA ≤ 60s) — we expect
a tighter distribution than other stops polled at 45s.

In [ ]:
sincai_lag = pd.to_numeric(df["sincai_eta_before"], errors="coerce").dropna()
sincai_lag.hist(bins=30, figsize=(8, 4),
                title="Gh. Sincai eta_before distribution (seconds)")
plt.xlabel("seconds")
plt.tight_layout()
plt.show()
print(sincai_lag.describe().round(1))

## Lag vs hour of day — is detection worse during peak hours?

In [ ]:
df["sincai_eta_before_s"] = pd.to_numeric(df["sincai_eta_before"], errors="coerce")
by_hour = df.groupby("hour")["sincai_eta_before_s"].mean()
by_hour.plot(kind="bar", figsize=(10, 4),
             title="Avg Gh. Sincai detection lag by hour", ylabel="seconds")
plt.tight_layout()
plt.show()